# 00 Environment check
Confirms the virtual environment, the device (CUDA, then MPS, then CPU) and the data keys.

In [ ]:
DEV_MODE = True    # True: small test run (dev: block of config.yaml). False: full experiment (full: block).

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from retinopath.core import load_config, get_device, seed_all, Timer
from retinopath import pipeline as PL

cfg = load_config(DEV_MODE); device = get_device(); seed_all(cfg.seed)
RES = cfg.P.out / "results"; FIG = cfg.P.out / "figures"; RES.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
print(f"DEV_MODE={DEV_MODE} | device={device} | outputs -> {cfg.P.out}")
if DEV_MODE: print("DEV_MODE: small test run. Its outputs are kept apart in outputs/dev and are not experiment results.")

In [ ]:
import sys, platform, shutil, importlib, torch
print("python", sys.version.split()[0], "|", platform.platform())
for m in ["numpy", "scipy", "pandas", "nibabel", "dipy", "torch", "h5py", "boto3", "statsmodels", "cvxpy"]:
    print(f"{m:12s}", importlib.import_module(m).__version__)
print("in .venv:", ".venv" in sys.prefix, "| free disk GB:", round(shutil.disk_usage(cfg.P.data).free / 1e9))
if device.type == "cuda":
    for i in range(torch.cuda.device_count()):
        print("GPU", i, torch.cuda.get_device_name(i), round(torch.cuda.mem_get_info(i)[1] / 1e9), "GB")
print("nodes in config:", list(cfg.nodes) or "none (single machine)")

In [ ]:
# Network forward and backward pass on one random patch (52 input channels = 3 shells x 15 SH + 7 anatomy)
from retinopath.model import build_model
m = build_model(cfg).to(device); x = torch.randn(1, 52, cfg.patch, cfg.patch, cfg.patch, device=device)
m(x).mean().backward(); print("network forward/backward OK on", device)

In [ ]:
# HCP keys present? (values are never printed)
from retinopath.download import read_env, hcp_client, s3_exists, HCP_BUCKET
env = read_env(cfg.P.env_file) if cfg.P.env_file.exists() else {}
print("keys in .env:", all(k in env for k in ("S3ACCESS_KEY", "S3SECRET_KEY")))
print("HCP bucket reachable:", s3_exists(hcp_client(cfg), HCP_BUCKET, "HCP_1200/100610/T1w/Diffusion/bvals"))